# MOO/EMO 10


Pfizer is one of the leading manufacturers and distributors of pharmaceuticals worldwide. The company's direct customers are physicians, who are regularly visited by Pfizer's sales representatives. This homework focuses only on a small piece of the company's operational activities, namely the Istanbul region in Turkey. In this area, 4 company representatives operate, and the area itself is divided into 22 smaller geographical units. Each representative has their permanent office in the center of one of the regions. For each such geographical unit, data is maintained regarding sales, the number of physicians and their profiles, and based on this data, an index is calculated that reflects the "effort" associated with operational activities in that region.

Pfizer seeks to assign to its representatives areas of operational activities in such a way that, first, each region is assigned to its representative and second, the total effort associated with regions assigned to each of the representatives is similar and close to 1. The last such assignment took place many years ago and currently the company is wondering whether, following the successive changes in region indicators that have occurred over time, a better allocation of its representatives is possible. The general tasks, criteria, considered in this problem are as follows:

1. **Distance Minimization $f_1$:** To make the travel of company representatives to assigned
operational activities efficient, the solution should in some way
minimize distances between the offices of representatives and the assigned regions
to them. Data regarding distances between their representatives' offices and
each of the regions is stored in the *distance.csv* file. The f1 function should be defined
as the sum of distances between each of the representative's offices and the regions assigned to them.

2. **Minimization of Changes $f_2$:** Pfizer wanted to avoid drastic changes in the distribution of operational areas to sales representatives. They, for example, have already built good relationships with customers and it would be inadvisable to disrupt these contacts. The current assignment of regions to representatives is recorded in the *assignment.csv* file. To define a measure of "changes", the model can count how many regions changed representatives (Note: only consider regions that were not previously assigned to a given representative but in the new solution are assigned to him). Furthermore, the components of this sum could be weighted by the "effort" of regions. This is justified in the sense that for regions with low effort, a potential change of the assigned representative is not as important as in a situation where the region had high effort. The effort coefficients for regions are stored in the *effortfulness.csv* file. Finally, $f_2$ is defined as a percentage – after multiplication by 100% – of deviations from the current assignment.

Given that the current assignment is not ideal, the criteria $f_1$ and $f_2$ are to some extent contradictory to each other. Therefore, the problem is multi-criteria in nature. Three constraints are imposed here:

1. Each region must be assigned exactly one representative of the company.
2. The sum of effort for regions assigned to each representative should
be in the interval $\\left[0.9, 1.1\\right]$. Currently, the total sum of effort for all
regions equals 4, so additional normalization is not required.
3. The recommended way to represent the problem solution is a binary matrix in
which 1 means that a given employee (column) is assigned a given region
(row). The decision variables should therefore have domain $\\left\\{0,1\\right\\}$.

Implement the solution using the PULP library for Python.

> **IMPORTANT**
>
> Code written in this notebook will be checked against automatic code checker and the points will be given based on its' results, please leave the function signatures unchanged.
> As a result there are no partial points for a programing tasks

In [1]:
import pandas as pd
from pulp import *

import matplotlib.pyplot as plt

D = pd.read_csv("distance.csv", index_col=0)
P = pd.read_csv("effortfulness.csv", index_col=0)
A = pd.read_csv("assignment.csv", index_col=0)


### Task 1 (maximum points:  1)



Using the PULP library, implement the criteria `f1` and `f2`. The variable `assignment` contains a list of lists of binary variables representing the new assignment.

In [12]:
def f1(assigment: list[list[LpVariable]], D: pd.DataFrame) -> LpAffineExpression:
    res = []
    for i in range(len(D.index)):
        for j in range(len(D.columns)):
            res.append(assigment[i][j] * D.iloc[i, j])
    return lpSum(res)

In [13]:
def f2(A: pd.DataFrame, new_assigment: list[list[LpVariable]], P: pd.DataFrame) -> LpAffineExpression:
    total_effort = P['Effortfulness'].sum()
    res = []
    for i in range(len(A.index)):
        for j in range(len(A.columns)):
            if A.iloc[i, j] == 0:
                res.append(new_assigment[i][j] * P.iloc[i, 0])
    return (lpSum(res) / total_effort) * 100

In [14]:
def goal_function(model: LpProblem, assignment: list[list[LpVariable]], D: pd.DataFrame) -> None:
    model += f1(assignment, D) 

### Task 2 (maximum points:  1)



Define a constraint that each region must be assigned exactly one representative

In [18]:
def single_representative_per_region_constraint(model, assigment):
    for i in range(len(assigment)):
        model += lpSum(assigment[i]) == 1

### Task 3 (maximum points:  1)



Define a constraint that each representative is assigned regions with similar effort

In [23]:
def similar_intesivity_constraint(model, assigment, P):
    for j in range(len(A.columns)):
        res = []
        for i in range(len(assigment)):
            res.append(assigment[i][j] * P.iloc[i, 0])
        model += lpSum(res) >= 0.9
        model += lpSum(res) <= 1.1

### Task 4 (maximum points:  1)



Define the epsilon constraint. If the value `epsilon` is equal to None, the constraint should not be added.

In [25]:
def epsilon_constraint(model, assigment, P, epsilon):
    if epsilon is None:
        return
    model += f2(A, assigment, P) <= epsilon

In [27]:
def create_model(
    D: pd.DataFrame,
    A: pd.DataFrame,
    P: pd.DataFrame,
    epsilon: float | None = None,
    debug: bool = False
) -> tuple[float, float] | None:
    model = LpProblem("Pfitzer")

    assigment = [
        [LpVariable(f"A_{i}_{j}", cat=const.LpBinary) for j in range(len(A.columns))] for i in range(len(A.index))
    ]

    goal_function(model, assigment, D)
    single_representative_per_region_constraint(model, assigment)
    similar_intesivity_constraint(model, assigment, P)
    epsilon_constraint(model, assigment, P, epsilon)

    v1 = f1(assigment, D)
    v2 = f2(A, assigment, P)

    model.solve(PULP_CBC_CMD(msg=0 if not debug else 1))

    return (v1.value(), v2.value()) if model.status == LpStatusOptimal else None

### Task 5 (maximum points:  1)



Write a function that will generate all feasible solutions for this problem. Solution generation should take no more than 1 second.

In [ ]:
# TODO
def generate_solutions(
    D: pd.DataFrame,
    A: pd.DataFrame,
    P: pd.DataFrame,
    debug: bool = False
) -> list[tuple[float, float]]:
    """None"""
    raise NotImplementedError()

In [ ]:
start = time()
repeat = 50

for _ in range(repeat):
    generate_solutions(D, A, P)

end = time()

print(f"Time: {(end - start) / repeat:.2f} seconds")

In [ ]:
solutions = generate_solutions(D, A, P)

plt.scatter(*zip(*solutions))
plt.show()

## NSGA-II

In [ ]:
import copy
import numpy as np

random = np.random.default_rng(0)


> IMPORTANT!
>
> When using random number generators, use only the generator defined in the random variable

In [ ]:
### Solution class
class Solution:
    def __init__(self, x: list[float], f: list[float], name: str) -> None:
        self.x = x ### decision variables
        self.f = f ### evaluation vector [f1, f2]
        self.front = 0 ## additional variable storing id of non-dominated front
        self.cd = 0.0 ## additional variable storing value of crowding distance
        self.name = name ### additional variable storing name of the solution

    def __str__(self) -> str:
        return f"[{self.name} : F = ({', '.join(f'{x:.4f}' for x in self.f)})]"

### Task 6 (maximum points:  1)



You need to fill a method for evaluation of solution basing on decision variable and the formula below:
$$\min f_1\left(\bold{x}\right) = x_1^{10}l$$
$$\min f_2\left(\bold{x}\right) = \left(1 - x_1^{10}\right)l$$
$$l = 1 + \left|x_2^{10}-0.5\right|+x_3+x_4+x_5$$
, where:  
$x_1, x_2 \in \left[0, 1\right]$  
$x_3, x_4, x_5 \in \left\{0, 1\right\}$

In [ ]:
# TODO
def evaluate(x: list[float]) -> list[float]:
    """None"""
    raise NotImplementedError()

In [ ]:
def constructInitialPopulation(N: int) -> list[Solution]:
    P = []
    for i in range(N):
        x = [j for j in range(5)]
        x[0] = random.random()
        x[1] = random.random()      
        for j in range(2,5): x[j] = random.integers(0,2).item()
        f = evaluate(x)
        P.append(Solution(x, f, str(0)+"-"+str(i)))  ### adding solution to list
        # which name consists of two parts: generation number
        # in which the solution was created and the number of solution in the population
    return P

P = constructInitialPopulation(10000)
for s in P[:10]: print(s) ### Wypisanie przykładowych rozwiązań (dla testu)


The function below plots solutions in P using a scatter plot. With large N values, you can observe how randomly generated solutions are distributed in the evaluation space. Please check, for example, with N=10000.

In [ ]:
def plotPopul(P: list[Solution]) -> None:
    plt.figure()
    X = [s.f[0] for s in P]
    Y = [s.f[1] for s in P]
    plt.plot(X, Y, ls='', marker='o')
    plt.xlim(0, 5.0);
    plt.ylim(0, 5.0)

plotPopul(P)

### Task 7 (maximum points:  1)



Justify the obtained distribution by referring to the definition of the problem (evaluation functions)

### Task 8 (maximum points:  1)



The function below should return a list of pairs of solution indices selected for reproduction. One offspring solution will be created from each pair. Selection should be based on a tournament of size 2. Output: list of pairs -> [ [idx11, idx12],...,[idxN1, idxN2] ]

In [ ]:
# TODO
def constructParents(N: int) -> list[list[int]]:
    """None"""
    raise NotImplementedError()

constructParents(10)

### Task 9 (maximum points:  1)



The function below should perform crossover of two input decision variable vectors. The procedure should implement the following scheme: the "offspring" determines the value of each variable by taking it from a random parent.

In [ ]:
# TODO
def getCrossed(xA: list[float], xB: list[float]) -> list[float]:
    """None"""
    raise NotImplementedError()

getCrossed([0.0, 0.5, 1.0], [1.0, 0.25, 0.75]) ### TEST

### Task 10 (maximum points:  1)



The function below should "mutate" the input vector x (operations should be performed) on x. The procedure should take into account the probability of mutation for each decision variable. For binary variables, mutation should be implemented as complementing the variable to 1: new x[i] = 1 - x[i]. For continuous variables, you can use Gaussian mutation, i.e. add a random number from a normal distribution with standard deviation given in the method parameter "std" to the value. You should protect the procedure against potential exceedance of the updated value beyond the allowed interval [0, 1]

In [ ]:
# TODO
def mutate(x: list[float], prob: float, std: float) -> None:
    """None"""
    raise NotImplementedError()

for std in [0.0, 0.1, 0.2]: ## TEST
    x = [0.0, 0.2, 1.0, 0.0, 0.0]
    mutate(x, 1.0, std)
    display(x)

In [ ]:
def constructOffspring(P: list[Solution], parents: list[list[int]], gen: int, std: float) -> list[Solution]:
    O = []
    prob = 1.0 / 5.0
    for i in range(len(parents)):
        xO = getCrossed(P[parents[i][0]].x, P[parents[i][1]].x)
        mutate(xO, prob, std)
        O.append(Solution(xO, evaluate(xO), str(gen) + "-" + str(i)))
    return O

### Task 11 (maximum points:  1)



The function below should assign solutions to non-dominated fronts. The output should be a list of lists of indices of solutions in P assigned to the respective fronts. For example, output: [[3,4,0],[1,5],[2]] means that solutions 0, 3, 4 in P are in the first front, 1 and 5 in the second, and solution 2 in the last.

In [ ]:
# TODO
def getNonDominatedFronts(P: list[Solution]) -> list[list[int]]:
    """None"""
    raise NotImplementedError()

In [ ]:
getNonDominatedFronts(P)

### Task 12 (maximum points:  1)



The method below should calculate the crowding distance (cd) values for solutions in P.
> **WARNING**:
>
> cd is calculated for non-dominated fronts separately; when calculating cd for solutions in one front, we "forget" about the remaining solutions. The method should return a vector of obtained cd values

In [ ]:
# TODO
def getCrowdingDistances(F: list[list[int]], P: list[Solution], normalize=True) -> list[float]:
    """None"""
    raise NotImplementedError()

In [ ]:
getCrowdingDistances(getNonDominatedFronts(P), P)


The method below assigns solutions their numbers of non-dominated fronts and cd values, and also sorts solutions in the population based on these values. If useCD = False, the cd measure is not taken into account during sorting. Testing the method for useCD = True and False will allow you to observe the benefit of using the cd measure.

In [ ]:
def applyScoresAndSort(P: list[Solution], useCD: bool = True) -> None:
    F = getNonDominatedFronts(P)
    CD = [0 for _ in P]
    if useCD:
        CD = getCrowdingDistances(F, P)

    for s, f in enumerate(F):
         for i in f: 
            P[i].front = s
            P[i].cd = CD[i]
   
    P.sort(key=lambda x: x.cd, reverse=True)
    P.sort(key=lambda x: x.front) # Kryterium sortowania

In [ ]:
applyScoresAndSort(P)

### Task 13 (maximum points:  1)



The function below should calculate auxiliary statistics that will help justify the use of the cd measure in the calculations. The method should return the mean, maximum, minimum value of cd, as well as the standard deviation of these values in the population P. When calculating statistics, ignore boundary values (infinity) obtained by boundary values. How should these measures be interpreted?

In [ ]:
# TODO
def getStatsCD(P: list[Solution]) -> tuple[float, float, float, float]:
    """None"""
    raise NotImplementedError()


The following two cells implement the NSGA-II algorithm, illustrate the constructed solutions at 5 different stages of the algorithm’s execution, and finally display the obtained statistics for CD. In the first cell, CD is not used during sorting; in the second one, it is. You should examine the code and test its behavior for different parameters (population sizes, etc.). For which variant did the algorithm perform better? (It should be the one that used the CD measure ;) )

In [ ]:
### Pominięcie miary CD
N = 20 # rozmiar populacji
GEN = 100 # liczba generacji/iteracji algorytmu
 
P = constructInitialPopulation(N) # utworzenie populacji początkowej
applyScoresAndSort(P, useCD = False) # posortowanie populacji wykorzystując fronty
# niezdominowane (bez miary cd; useCD = False)

for gen in range(GEN): # iteracja po kolejnych generacjach
    C = constructParents(N) # skonstruowanie identyfikatorów rodziców
    O = constructOffspring(P, C, gen + 1, 0.1) # konstrukcja potomstwa
    M = P + O # połączenie obecnej populacji z potomną
    applyScoresAndSort(M, useCD = False) # posortowanie tak połączonej populacji
    P = M[:N] # "przeżywają najlepiej przystosowani"
    if gen % (GEN / 4) == 0 or gen == GEN - 1: plotPopul(P) # co jakiś krok
        # ilustrowana jest populacja

# wypisanie statystyk
print(getStatsCD(P))

In [ ]:
N = 20
GEN = 100

P = constructInitialPopulation(N)
applyScoresAndSort(P, useCD = True)

for gen in range(GEN):
    C = constructParents(N)
    O = constructOffspring(P, C, gen + 1, 0.1)
    M = P + O
    applyScoresAndSort(M, useCD = True)
    P = M[:N]
    if gen % (GEN / 5) == 0 or gen == GEN - 1: plotPopul(P)

print(getStatsCD(P))

### Task 14 (maximum points:  2)


<b>Dla chętnych: </b> algorytmy ewolucyjne są losowe. Z tego powodu jednokrotne ich uruchomienie nie jest wiarygodne. By uzyskać bardziej wiarygodny wynik, algorytm może zostać uruchomiony np. 20 razy a uzyskane statystyki uśrednione. Dodatkowo można je obliczyć dla każdej generacji (nie tylko w ostatniej) i wykreślić ich zbieżność na wykresie liniowym. Na jednym wykresie można wykreślić wyniki uzyskane przez oba warianty algorytmu - dzięki czemu różnice powinny byc lepiej widoczne. Proszę zauważyć jednak, że statystki dla cd nie są możliwie najlepszą miarą oceny jakości działania algorytmu. Na samym początku, gdy rozwiązania są silnie rozproszone, ich wartości cd powinny być bardzo wysokie, co jest w sprzeczności z założeniem, że "im większe cd tym lepiej". Aby wiarygodniej oceniać takie algorytmy, należy także wykorzystywać miary, które szacują odległość rozwiązań do frontu Pareta. Można zaproponować taką miarę i wykorzystać ją do pokazania "bliskości" populacji do tego frontu. Dopiero jednoczesne zestawienie takiej miary i miary oceniającej rozproszenie rozwiązań umożliwi lepsze porównanie algorytmów. 

### Task 15 (maximum points:  2)



**For the interested:** evolutionary algorithms are random. For this reason, a single run is not reliable. The algorithm can be run, for example, 20 times and the resulting statistics averaged. Additionally, they can be calculated for each generation (not just the last one) and their convergence plotted on a line chart. On one chart, you can plot the results obtained by both variants of the algorithm - so the differences should be more visible. However, please note that statistics for cd are not necessarily the best measure of algorithm quality. At the beginning, when solutions are highly dispersed, their cd values should be very high, which is contrary to the assumption that "the higher the cd the better". To more reliably evaluate such algorithms, measures that estimate the distance of solutions to the Pareto front should also be used. You can propose such a measure and use it to show the "proximity" of the population to that front. Only a simultaneous set of such a measure and a measure assessing the dispersion of solutions will allow for a better comparison of algorithms.